# Faz 2 — Veri Kalitesi, Gelişmiş MVI ve Sızıntısız Ön İşleme

**Sorumlu:** Ümit (Veri Mimarı ve Kalite Uzmanı)

Bu notebook proje şartnamesindeki Faz 2 gereksinimlerini uygular:
1. **Veri Kalitesi ve Geçerlilik Kontrolleri:** Kopyalar (`duplicates`), aykırı değerler (`outliers`), geçersiz kayıtlar ve yapısal anomaliler (`structural anomalies`).
2. **Sızıntısız Train/Test Ayrımı ve Sütun Eleme:** Sabit ve aşırı eksik (`>%50`) sütunların yalnızca `X_train` istatistiklerine bakılarak elenmesi.
3. **Eksik Veri Alt Sınırı (≥ %5 NaN):** Eğitim seti referans alınarak `%6` seviyesine sistematik NaN enjeksiyonu.
4. **Gelişmiş MVI Karşılaştırması:** Basit yöntem (`SimpleImputer`: median/mode) ile gelişmiş yöntemin (`IterativeImputer / MICE`) gizlenen gerçek test hücreleri üzerinde RMSE ve MAE ile karşılaştırılması.
5. **Veri Sızıntısız (`Leakage-Free`) Encoding & Scaling:** `Pipeline` ve `ColumnTransformer` ile yalnızca `X_train` üzerinden öğrenen `QuantileClipper`, `StandardScaler`, `TargetEncoder` (+ `StandardScaler`) ve `OneHotEncoder` mimarisi.

In [1]:
import time

import pandas as pd
from sklearn.model_selection import train_test_split

from src import config
from src import preprocessing as pp
from src.data_loader import load_raw_data

## 1. Veri Kalitesi, Kopya, Geçerlilik ve Aykırı Değer Analizi

In [2]:
raw_df = load_raw_data()
pp.quality_report(raw_df)

,Kontrol,Sonuç
0,Tekrarlı encounter_id,0
1,Tam kopya satır (ID hariç),0
2,Geçersiz cinsiyet ('Unknown/Invalid'),3
3,"Bilinmeyen kod: admission_type_id [5, 6, 8]",10396
4,"Bilinmeyen kod: discharge_disposition_id [18, ...",4680
5,"Bilinmeyen kod: admission_source_id [9, 15, 17...",7067
6,Sabit (tek değerli) sütun,"examide, citoglipton"
7,%50 üstü eksik sütun,weight


In [3]:
pp.outlier_report(raw_df).round(4)

,Sütun,Alt sınır,Üst sınır,Aykırı,Oran
0,time_in_hospital,-4.0,12.0,2252,0.0221
1,num_lab_procedures,-8.0,96.0,143,0.0014
2,num_procedures,-3.0,5.0,4954,0.0487
3,num_medications,-5.0,35.0,2557,0.0251
4,number_outpatient,0.0,0.0,16739,0.1645
5,number_emergency,0.0,0.0,11383,0.1119
6,number_inpatient,-1.5,2.5,7049,0.0693
7,number_diagnoses,1.5,13.5,281,0.0028


## 2. Sızıntısız Train/Test Ayrımı ve Sütun Eleme

Satır bazlı kopya ve şema temizliği (`clean_records`) sonrasında veri `%80` eğitim ve `%20` test olarak katmanlı biçimde ayrılır. Sabit (`nunique <= 1`) ve aşırı eksik (`>%50`) sütunların hangileri olduğu **yalnızca eğitim seti (`train_idx`)** üzerinden belirlenir.

In [4]:
df_records = pp.clean_records(raw_df)
train_idx, test_idx = train_test_split(
    df_records.index,
    test_size=0.20,
    stratify=df_records[config.TARGET],
    random_state=config.RANDOM_STATE,
)

dropped_cols = pp.find_invalid_columns(df_records.loc[train_idx])
df_clean = df_records.drop(columns=dropped_cols)

print(f'Eğitim setine göre elenen sütunlar : {dropped_cols}')
print(f'Temizlik sonrası boyut             : {df_clean.shape}')
print(f'Kalan özellik sayısı               : '
      f'{df_clean.shape[1] - 1}')
print(f'Doğal eksik veri oranı (eğitim)    : '
      f'{pp.missing_ratio(df_clean.loc[train_idx]):.2%}')

Eğitim setine göre elenen sütunlar : ['acetohexamide', 'citoglipton', 'examide', 'troglitazone', 'weight']
Temizlik sonrası boyut             : (101766, 43)
Kalan özellik sayısı               : 42
Doğal eksik veri oranı (eğitim)    : 2.72%


## 3. Sistematik Eksik Veri Enjeksiyonu (≥ %5 NaN Şartı)

Enjeksiyon oranı yalnızca `train_idx` eksik veri oranından hesaplanır ve tekrarlanabilir (`random_state=42`) şekilde uygulanır.

In [5]:
df_missing, injection_mask = pp.inject_missing(
    df_clean, train_index=train_idx
)
train_missing = pp.missing_ratio(df_missing.loc[train_idx])
total_missing = pp.missing_ratio(df_missing)
n_injected = int(injection_mask.to_numpy().sum())

print(f'Eğitim seti toplam NaN oranı : {train_missing:.2%}')
print(f'Genel toplam NaN oranı       : {total_missing:.2%}')
print(f'Enjekte edilen hücre sayısı  : {n_injected:,}')
assert train_missing >= config.MIN_MISSING_RATIO
assert total_missing >= config.MIN_MISSING_RATIO

Eğitim seti toplam NaN oranı : 6.00%
Genel toplam NaN oranı       : 6.00%
Enjekte edilen hücre sayısı  : 140,124


## 4. Sızıntısız MVI Karşılaştırması (Baseline vs. MICE)

In [6]:
X_train = df_missing.loc[train_idx].drop(columns=config.TARGET)
y_train = df_missing.loc[train_idx, config.TARGET]
X_test = df_missing.loc[test_idx].drop(columns=config.TARGET)
y_test = df_missing.loc[test_idx, config.TARGET]

X_test_true = df_clean.loc[test_idx].drop(columns=config.TARGET)
test_mask = injection_mask.loc[test_idx]

print(f'X_train boyutu: {X_train.shape} | Pozitif oranı: '
      f'{y_train.mean():.4f}')
print(f'X_test  boyutu: {X_test.shape}  | Pozitif oranı: '
      f'{y_test.mean():.4f}')

X_train boyutu: (81412, 42) | Pozitif oranı: 0.1116
X_test  boyutu: (20354, 42)  | Pozitif oranı: 0.1116


In [7]:
methods = {
    'SimpleImputer (Baseline - Median/Mode)': 'simple',
    'IterativeImputer / MICE (Advanced MVI)': 'mice',
}

fitted_pipelines = {}
detailed_tables = {}
summary_rows = []

for label, key in methods.items():
    t0 = time.perf_counter()
    pipe = pp.build_preprocessor(df_missing, imputer=key)
    pipe.fit(X_train, y_train)
    eval_df = pp.evaluate_imputation(
        pipe, X_test_true, X_test, test_mask
    )
    elapsed = time.perf_counter() - t0
    fitted_pipelines[key] = pipe
    detailed_tables[key] = eval_df
    tot = eval_df.iloc[-1]
    summary_rows.append((
        label, int(tot['Enjekte NaN']),
        tot['RMSE'], tot['MAE'], elapsed,
    ))

summary_df = pd.DataFrame(
    summary_rows,
    columns=['Yöntem', 'Test Enjekte NaN', 'RMSE', 'MAE', 'Süre (sn)'],
)
summary_df.round(4)

,Yöntem,Test Enjekte NaN,RMSE,MAE,Süre (sn)
0,SimpleImputer (Baseline - Median/Mode),28230,1.0328,0.6312,1.1669
1,IterativeImputer / MICE (Advanced MVI),28230,0.8711,0.6202,136.5123


### Sütun Bazında Baseline vs. MICE Hata Karşılaştırması

In [8]:
comparison_cols = detailed_tables['simple'][['Sütun', 'Enjekte NaN']].copy()
comparison_cols['Simple_RMSE'] = detailed_tables['simple']['RMSE']
comparison_cols['MICE_RMSE'] = detailed_tables['mice']['RMSE']
comparison_cols['Simple_MAE'] = detailed_tables['simple']['MAE']
comparison_cols['MICE_MAE'] = detailed_tables['mice']['MAE']
comparison_cols['RMSE_İyileşme_%'] = (
    100.0 * (comparison_cols['Simple_RMSE']
             - comparison_cols['MICE_RMSE'])
    / comparison_cols['Simple_RMSE']
)
comparison_cols.round(4)

,Sütun,Enjekte NaN,Simple_RMSE,MICE_RMSE,Simple_MAE,MICE_MAE,RMSE_İyileşme_%
0,time_in_hospital,3497,0.9975,0.8080,0.7526,0.6231,18.9975
1,num_lab_procedures,3525,0.9982,0.8558,0.7904,0.6724,14.2629
2,num_procedures,3529,1.0267,0.8833,0.7431,0.6831,13.9698
3,num_medications,3589,1.0461,0.7890,0.7820,0.6076,24.5764
4,number_outpatient,3604,1.0232,0.9375,0.3445,0.5486,8.3812
5,number_emergency,3474,0.9963,0.9098,0.2948,0.5112,8.6751
6,number_inpatient,3527,1.1290,0.9246,0.5370,0.6288,18.1092
7,number_diagnoses,3485,1.0383,0.8490,0.8070,0.6878,18.2338
8,TOPLAM,28230,1.0328,0.8711,0.6312,0.6202,15.6565


## 5. Seçilen Boru Hattı (MICE) ile Dönüştürme ve Kayıt

Hem RMSE hem MAE metriklerinde en düşük hatayı üreten `IterativeImputer / MICE` boru hattı ile eğitim ve test setleri dönüştürülerek sonraki fazlar için `data/processed/` dizinine kaydedilir.

In [9]:
best_pipe = fitted_pipelines['mice']
X_train_proc = pp.transform_to_dataframe(best_pipe, X_train)
X_test_proc = pp.transform_to_dataframe(best_pipe, X_test)

assert X_train_proc.isna().sum().sum() == 0
assert X_test_proc.isna().sum().sum() == 0

scaled_cols = pp.NUMERIC_COLUMNS + pp.HIGH_CARDINALITY_COLUMNS
print(f'İşlenmiş X_train boyutu: {X_train_proc.shape}')
print(f'İşlenmiş X_test  boyutu: {X_test_proc.shape}')
print('\nÖlçeklenen sayısal + TargetEncoded sütunlar (mean/std):')
X_train_proc[scaled_cols].agg(['mean', 'std']).T.round(4)

İşlenmiş X_train boyutu: (81412, 154)
İşlenmiş X_test  boyutu: (20354, 154)

Ölçeklenen sayısal + TargetEncoded sütunlar (mean/std):


,mean,std
time_in_hospital,0.0002,0.9440
num_lab_procedures,0.0001,0.9385
num_procedures,-0.0003,0.9341
num_medications,-0.0006,0.9546
number_outpatient,-0.0002,0.9168
number_emergency,-0.0001,0.9251
number_inpatient,0.0007,0.9260
number_diagnoses,0.0007,0.9351
diag_1,-0.0007,0.9492
diag_2,0.0031,0.9746


In [10]:
config.PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

df_missing.to_pickle(
    config.PROCESSED_DATA_DIR / 'df_missing.pkl'
)
X_train_proc.to_pickle(
    config.PROCESSED_DATA_DIR / 'X_train.pkl'
)
X_test_proc.to_pickle(
    config.PROCESSED_DATA_DIR / 'X_test.pkl'
)
y_train.to_pickle(
    config.PROCESSED_DATA_DIR / 'y_train.pkl'
)
y_test.to_pickle(
    config.PROCESSED_DATA_DIR / 'y_test.pkl'
)
print('Kaydedilen dosyalar:',
      [p.name for p in sorted(config.PROCESSED_DATA_DIR.glob('*.pkl'))])

Kaydedilen dosyalar: ['df_missing.pkl', 'X_test.pkl', 'X_train.pkl', 'y_test.pkl', 'y_train.pkl']
